In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams.update({
    "figure.figsize": (12, 7),
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.20,
    "font.size": 11
})

OPT_COLORS = {
    "SGD": "#ff4d6d",
    "AdaGrad": "#00b4d8",
    "RMSProp": "#7b2cbf",
    "Adam": "#f4a261"
}

print("TensorFlow:", tf.__version__)

In [ ]:
X, y = make_regression(
    n_samples=260,
    n_features=1,
    n_informative=1,
    noise=10.0,
    bias=18.0,
    random_state=42
)

X = X.astype(np.float32)
y = y.astype(np.float32).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train).astype(np.float32)
X_test_s = x_scaler.transform(X_test).astype(np.float32)

y_train_s = y_scaler.fit_transform(y_train).astype(np.float32)
y_test_s = y_scaler.transform(y_test).astype(np.float32)

data_summary = pd.DataFrame({
    "Split": ["Train", "Test"],
    "Samples": [len(X_train_s), len(X_test_s)],
    "X mean": [X_train_s.mean(), X_test_s.mean()],
    "X std": [X_train_s.std(), X_test_s.std()],
    "y mean": [y_train_s.mean(), y_test_s.mean()],
    "y std": [y_train_s.std(), y_test_s.std()]
})

display(data_summary.round(4))

fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(
    X_train.ravel(),
    y_train.ravel(),
    s=55,
    alpha=0.55,
    label="Train"
)

ax.scatter(
    X_test.ravel(),
    y_test.ravel(),
    s=85,
    marker="D",
    alpha=0.90,
    label="Test"
)

ax.set_title("Linear Dataset Created with sklearn")
ax.set_xlabel("Original X")
ax.set_ylabel("Original y")
ax.legend()

plt.show()

In [ ]:
def build_linear_ann():
    return keras.Sequential([
        keras.layers.Input(shape=(1,)),
        keras.layers.Dense(1, activation=None, name="linear_neuron")
    ])

tf.keras.backend.clear_session()
tf.random.set_seed(123)

base_model = build_linear_ann()
_ = base_model(X_train_s[:1])

initial_kernel = np.array([[0.15]], dtype=np.float32)
initial_bias = np.array([1.80], dtype=np.float32)

initial_weights = [initial_kernel, initial_bias]

def fresh_model():
    model = build_linear_ann()
    model(X_train_s[:1])
    model.set_weights([w.copy() for w in initial_weights])
    return model

models = {
    "SGD": fresh_model(),
    "AdaGrad": fresh_model(),
    "RMSProp": fresh_model(),
    "Adam": fresh_model()
}

for name, model in models.items():
    print(name, "initial weights:", [w.ravel().tolist() for w in model.get_weights()])

In [ ]:
optimizer_settings = {
    "SGD": tf.keras.optimizers.SGD(
        learning_rate=0.08
    ),
    "AdaGrad": tf.keras.optimizers.Adagrad(
        learning_rate=0.35,
        initial_accumulator_value=0.0
    ),
    "RMSProp": tf.keras.optimizers.RMSprop(
        learning_rate=0.06,
        rho=0.90
    ),
    "Adam": tf.keras.optimizers.Adam(
        learning_rate=0.05,
        beta_1=0.90,
        beta_2=0.999,
        epsilon=1e-7
    )
}

steps = 100
histories = {}

for name, model in models.items():
    optimizer = optimizer_settings[name]
    records = []

    for step in range(steps + 1):
        w, b = model.get_weights()

        with tf.GradientTape() as tape:
            prediction = model(X_train_s, training=True)
            step_loss = tf.reduce_mean(tf.square(prediction - y_train_s))

        grads = tape.gradient(step_loss, model.trainable_variables)

        grad_w = float(grads[0].numpy().ravel()[0])
        grad_b = float(grads[1].numpy().ravel()[0])

        if step == 0:
            records.append({
                "step": 0,
                "w": float(w.ravel()[0]),
                "b": float(b.ravel()[0]),
                "loss": float(step_loss.numpy()),
                "grad_w": grad_w,
                "grad_b": grad_b,
                "old_w": float(w.ravel()[0]),
                "old_b": float(b.ravel()[0])
            })
            continue

        old_w = float(w.ravel()[0])
        old_b = float(b.ravel()[0])

        optimizer.apply_gradients(
            zip(grads, model.trainable_variables)
        )

        new_w, new_b = model.get_weights()

        new_loss = tf.reduce_mean(
            tf.square(
                model(X_train_s, training=False) - y_train_s
            )
        )

        records.append({
            "step": step,
            "w": float(new_w.ravel()[0]),
            "b": float(new_b.ravel()[0]),
            "loss": float(new_loss.numpy()),
            "grad_w": grad_w,
            "grad_b": grad_b,
            "old_w": old_w,
            "old_b": old_b
        })

    histories[name] = pd.DataFrame(records)

for name in histories:
    display(histories[name].head(5))

In [ ]:
def reconstruct_optimizer_state(history, name):
    gw = history["grad_w"].to_numpy()
    gb = history["grad_b"].to_numpy()

    gradients = np.column_stack([gw, gb])

    state_1 = np.zeros_like(gradients)
    state_2 = np.zeros_like(gradients)

    effective_lr = np.zeros_like(gradients)
    momentum_direction = np.zeros_like(gradients)

    if name == "SGD":
        effective_lr[:] = 0.08
        momentum_direction[:] = gradients

    elif name == "AdaGrad":
        for i in range(1, len(history)):
            state_1[i] = state_1[i-1] + gradients[i]**2

        effective_lr = 0.35 / (
            np.sqrt(state_1) + 1e-7
        )

    elif name == "RMSProp":
        rho = 0.90

        for i in range(1, len(history)):
            state_2[i] = (
                rho * state_2[i-1]
                + (1-rho) * gradients[i]**2
            )

        effective_lr = 0.06 / (
            np.sqrt(state_2) + 1e-7
        )

    elif name == "Adam":
        beta1 = 0.90
        beta2 = 0.999
        eps = 1e-7
        lr = 0.05

        for i in range(1, len(history)):
            state_1[i] = (
                beta1 * state_1[i-1]
                + (1-beta1) * gradients[i]
            )

            state_2[i] = (
                beta2 * state_2[i-1]
                + (1-beta2) * gradients[i]**2
            )

        for i in range(1, len(history)):
            t = i

            m_hat = state_1[i] / (
                1 - beta1**t
            )

            v_hat = state_2[i] / (
                1 - beta2**t
            )

            momentum_direction[i] = (
                m_hat / (np.sqrt(v_hat) + eps)
            )

        effective_lr[:] = lr
        effective_lr[0] = 0.0

    return {
        "gradients": gradients,
        "state_1": state_1,
        "state_2": state_2,
        "effective_lr": effective_lr,
        "momentum_direction": momentum_direction
    }

states = {}

for name in histories:
    states[name] = reconstruct_optimizer_state(
        histories[name],
        name
    )

print("Optimizer state tracking ready.")

In [ ]:
def mse_at(w, b, X=X_train_s, y=y_train_s):
    pred = w * X + b
    return np.mean((pred - y)**2)

w_min, w_max = -1.4, 2.5
b_min, b_max = -2.4, 2.4

w_grid = np.linspace(w_min, w_max, 180)
b_grid = np.linspace(b_min, b_max, 180)

WG, BG = np.meshgrid(w_grid, b_grid)
ZG = np.zeros_like(WG)

for i in range(WG.shape[0]):
    ZG[i] = mse_at(WG[i], BG[i])

levels = np.linspace(
    np.percentile(ZG, 2),
    np.percentile(ZG, 97),
    45
)

print("Loss surface shape:", ZG.shape)

In [ ]:
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection="3d")

surface = ax.plot_surface(
    WG,
    BG,
    ZG,
    cmap="turbo",
    alpha=0.78,
    linewidth=0,
    antialiased=True
)

for name in histories:
    h = histories[name]

    ax.plot(
        h["w"],
        h["b"],
        h["loss"],
        linewidth=3.2,
        color=OPT_COLORS[name],
        label=name
    )

    ax.scatter(
        h["w"].iloc[0],
        h["b"].iloc[0],
        h["loss"].iloc[0],
        s=90,
        color=OPT_COLORS[name],
        edgecolor="black"
    )

    ax.scatter(
        h["w"].iloc[-1],
        h["b"].iloc[-1],
        h["loss"].iloc[-1],
        s=120,
        marker="*",
        color=OPT_COLORS[name],
        edgecolor="black"
    )

ax.set_title(
    "3D Loss Landscape: Four Optimizer Trajectories",
    fontsize=18,
    fontweight="bold"
)

ax.set_xlabel("Weight w")
ax.set_ylabel("Bias b")
ax.set_zlabel("MSE")

fig.colorbar(
    surface,
    ax=ax,
    shrink=0.62,
    pad=0.08,
    label="MSE"
)

ax.legend()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

filled = ax.contourf(
    WG,
    BG,
    ZG,
    levels=levels,
    cmap="turbo",
    alpha=0.95
)

ax.contour(
    WG,
    BG,
    ZG,
    levels=levels[::2],
    colors="white",
    linewidths=0.55,
    alpha=0.35
)

for name in histories:
    h = histories[name]

    ax.plot(
        h["w"],
        h["b"],
        color=OPT_COLORS[name],
        linewidth=3,
        label=name,
        marker="o",
        markevery=12,
        markersize=4
    )

    ax.scatter(
        h["w"].iloc[0],
        h["b"].iloc[0],
        s=120,
        color=OPT_COLORS[name],
        edgecolor="black"
    )

    ax.scatter(
        h["w"].iloc[-1],
        h["b"].iloc[-1],
        s=190,
        color=OPT_COLORS[name],
        marker="*",
        edgecolor="black",
        zorder=5
    )

ax.scatter(
    WG.ravel()[np.argmin(ZG.ravel())],
    BG.ravel()[np.argmin(ZG.ravel())],
    s=240,
    marker="X",
    color="black",
    label="Lowest grid loss",
    zorder=6
)

ax.set_title(
    "Contour Map: How SGD, AdaGrad, RMSProp and Adam Travel",
    fontsize=18,
    fontweight="bold"
)

ax.set_xlabel("Weight w")
ax.set_ylabel("Bias b")

fig.colorbar(
    filled,
    ax=ax,
    label="MSE"
)

ax.legend()

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

for name in histories:
    h = histories[name]

    axes[0].plot(
        h["step"],
        h["loss"],
        linewidth=3,
        color=OPT_COLORS[name],
        label=name
    )

    axes[1].semilogy(
        h["step"],
        h["loss"] + 1e-10,
        linewidth=3,
        color=OPT_COLORS[name],
        label=name
    )

axes[0].set_title("Training Loss")
axes[0].set_xlabel("Optimization step")
axes[0].set_ylabel("MSE")
axes[0].legend()

axes[1].set_title("Training Loss — Log Scale")
axes[1].set_xlabel("Optimization step")
axes[1].set_ylabel("MSE (log)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.ravel()

for ax, name in zip(axes, histories):
    state = states[name]
    lr = state["effective_lr"]

    if name == "SGD":
        ax.plot(
            lr[:, 0],
            linewidth=3,
            color=OPT_COLORS[name],
            label="w LR"
        )
        ax.plot(
            lr[:, 1],
            linewidth=3,
            linestyle="--",
            color="#22223b",
            label="b LR"
        )

    elif name in ["AdaGrad", "RMSProp"]:
        ax.plot(
            lr[:, 0],
            linewidth=3,
            color=OPT_COLORS[name],
            label="w effective LR"
        )
        ax.plot(
            lr[:, 1],
            linewidth=3,
            linestyle="--",
            color="#22223b",
            label="b effective LR"
        )

    else:
        ax.plot(
            state["momentum_direction"][:, 0],
            linewidth=3,
            color=OPT_COLORS[name],
            label="Adam normalized direction: w"
        )
        ax.plot(
            state["momentum_direction"][:, 1],
            linewidth=3,
            linestyle="--",
            color="#22223b",
            label="Adam normalized direction: b"
        )

    ax.set_title(
        f"{name}: Optimizer Mechanics"
    )
    ax.set_xlabel("Step")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

for name in histories:
    h = histories[name]

    axes[0].plot(
        h["step"],
        h["grad_w"],
        linewidth=3,
        color=OPT_COLORS[name],
        label=name
    )

    axes[1].plot(
        h["step"],
        h["grad_b"],
        linewidth=3,
        color=OPT_COLORS[name],
        label=name
    )

axes[0].axhline(0, color="black", linewidth=1)
axes[1].axhline(0, color="black", linewidth=1)

axes[0].set_title("Gradient of Weight w")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Gradient")
axes[0].legend()

axes[1].set_title("Gradient of Bias b")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Gradient")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
def contour_animation(name, interval=90):
    h = histories[name]
    path = h[["w", "b"]].to_numpy()

    fig, ax = plt.subplots(figsize=(12, 9))

    ax.contourf(
        WG,
        BG,
        ZG,
        levels=levels,
        cmap="turbo",
        alpha=0.95
    )

    ax.contour(
        WG,
        BG,
        ZG,
        levels=levels[::2],
        colors="white",
        linewidths=0.45,
        alpha=0.30
    )

    line, = ax.plot(
        [],
        [],
        color=OPT_COLORS[name],
        linewidth=4
    )

    point, = ax.plot(
        [],
        [],
        marker="o",
        markersize=13,
        color=OPT_COLORS[name],
        markeredgecolor="black"
    )

    title = ax.set_title("")
    info = ax.text(
        0.03,
        0.96,
        "",
        transform=ax.transAxes,
        va="top",
        fontsize=12,
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.85
        )
    )

    ax.set_xlabel("Weight w")
    ax.set_ylabel("Bias b")

    def update(frame):
        current = path[:frame+1]

        line.set_data(
            current[:, 0],
            current[:, 1]
        )

        point.set_data(
            [current[-1, 0]],
            [current[-1, 1]]
        )

        row = h.iloc[frame]

        title.set_text(
            f"{name} | Step {int(row['step'])}"
        )

        info.set_text(
            f"w = {row['w']:.5f}\n"
            f"b = {row['b']:.5f}\n"
            f"Loss = {row['loss']:.6f}\n"
            f"grad_w = {row['grad_w']:+.5f}\n"
            f"grad_b = {row['grad_b']:+.5f}"
        )

        return line, point, title, info

    anim = FuncAnimation(
        fig,
        update,
        frames=len(path),
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)

    return HTML(anim.to_jshtml())

In [ ]:
display(contour_animation("SGD"))

In [ ]:
display(contour_animation("AdaGrad"))

In [ ]:
display(contour_animation("RMSProp"))

In [ ]:
display(contour_animation("Adam"))

In [ ]:
def mechanics_animation(name, interval=140):

    h = histories[name].reset_index(drop=True)
    state = states[name]

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(19, 6)
    )

    ax0, ax1, ax2 = axes

    ax0.contourf(
        WG,
        BG,
        ZG,
        levels=levels,
        cmap="turbo",
        alpha=0.92
    )

    ax0.contour(
        WG,
        BG,
        ZG,
        levels=levels[::2],
        colors="white",
        linewidths=0.45,
        alpha=0.30
    )

    path_line, = ax0.plot(
        [],
        [],
        color=OPT_COLORS[name],
        linewidth=4
    )

    path_point, = ax0.plot(
        [],
        [],
        marker="o",
        markersize=12,
        color=OPT_COLORS[name],
        markeredgecolor="black"
    )

    ax0.set_xlim(WG.min(), WG.max())
    ax0.set_ylim(BG.min(), BG.max())
    ax0.set_title(f"{name} — Optimizer Path")
    ax0.set_xlabel("Weight (w)")
    ax0.set_ylabel("Bias (b)")

    xbar = np.arange(4)

    bars = ax1.bar(
        xbar,
        np.zeros(4),
        color=[
            OPT_COLORS[name],
            OPT_COLORS[name],
            "#90be6d",
            "#f4a261"
        ],
        alpha=0.9
    )

    ax1.set_xticks(xbar)
    ax1.set_xticklabels(
        [
            "Gradient w",
            "Gradient b",
            "State w",
            "State b"
        ],
        rotation=20
    )

    ax1.set_title("Current Step Ingredients")
    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax2.axis("off")

    info = ax2.text(
        0.02,
        0.98,
        "",
        transform=ax2.transAxes,
        va="top",
        ha="left",
        family="monospace",
        fontsize=11,
        bbox=dict(
            boxstyle="round,pad=0.7",
            facecolor="white",
            alpha=0.92
        )
    )

    def update(frame):

        row = h.iloc[frame]

        path = h.iloc[:frame + 1]

        path_line.set_data(
            path["w"].to_numpy(),
            path["b"].to_numpy()
        )

        path_point.set_data(
            [row["w"]],
            [row["b"]]
        )

        grad_w = float(
            state["gradients"][frame, 0]
        )

        grad_b = float(
            state["gradients"][frame, 1]
        )

        if name == "SGD":

            values = [
                grad_w,
                grad_b,
                0.0,
                0.0
            ]

            rule = "theta_new = theta - eta*g"

            extra = (
                f"\n"
                f"learning rate : 0.08"
            )

        elif name == "AdaGrad":

            state_w = float(
                state["state_1"][frame, 0]
            )

            state_b = float(
                state["state_1"][frame, 1]
            )

            values = [
                grad_w,
                grad_b,
                state_w,
                state_b
            ]

            lr_w = float(
                state["effective_lr"][frame, 0]
            )

            lr_b = float(
                state["effective_lr"][frame, 1]
            )

            rule = "G = G + g²"

            extra = (
                f"\n"
                f"G(w)           : {state_w:.6f}\n"
                f"G(b)           : {state_b:.6f}\n"
                f"effective LR(w): {lr_w:.6f}\n"
                f"effective LR(b): {lr_b:.6f}"
            )

        elif name == "RMSProp":

            state_w = float(
                state["state_2"][frame, 0]
            )

            state_b = float(
                state["state_2"][frame, 1]
            )

            values = [
                grad_w,
                grad_b,
                state_w,
                state_b
            ]

            lr_w = float(
                state["effective_lr"][frame, 0]
            )

            lr_b = float(
                state["effective_lr"][frame, 1]
            )

            rule = "v = rho*v + (1-rho)*g²"

            extra = (
                f"\n"
                f"v(w)           : {state_w:.6f}\n"
                f"v(b)           : {state_b:.6f}\n"
                f"effective LR(w): {lr_w:.6f}\n"
                f"effective LR(b): {lr_b:.6f}"
            )

        elif name == "Adam":

            m_w = float(
                state["state_1"][frame, 0]
            )

            m_b = float(
                state["state_1"][frame, 1]
            )

            v_w = float(
                state["state_2"][frame, 0]
            )

            v_b = float(
                state["state_2"][frame, 1]
            )

            values = [
                grad_w,
                grad_b,
                m_w,
                v_w
            ]

            step = int(row["step"])

            if step > 0:

                beta1 = 0.90
                beta2 = 0.999
                eps = 1e-7

                m_hat_w = (
                    m_w /
                    (1 - beta1 ** step)
                )

                m_hat_b = (
                    m_b /
                    (1 - beta1 ** step)
                )

                v_hat_w = (
                    v_w /
                    (1 - beta2 ** step)
                )

                v_hat_b = (
                    v_b /
                    (1 - beta2 ** step)
                )

                adam_direction_w = (
                    m_hat_w /
                    (np.sqrt(v_hat_w) + eps)
                )

                adam_direction_b = (
                    m_hat_b /
                    (np.sqrt(v_hat_b) + eps)
                )

            else:

                m_hat_w = 0.0
                m_hat_b = 0.0
                v_hat_w = 0.0
                v_hat_b = 0.0
                adam_direction_w = 0.0
                adam_direction_b = 0.0

            rule = "m + v + bias correction"

            extra = (
                f"\n"
                f"m(w)           : {m_w:+.6f}\n"
                f"m(b)           : {m_b:+.6f}\n"
                f"v(w)           : {v_w:.6f}\n"
                f"v(b)           : {v_b:.6f}\n"
                f"m_hat(w)       : {m_hat_w:+.6f}\n"
                f"m_hat(b)       : {m_hat_b:+.6f}\n"
                f"v_hat(w)       : {v_hat_w:.6f}\n"
                f"v_hat(b)       : {v_hat_b:.6f}\n"
                f"Adam direction : {adam_direction_w:+.6f}"
            )

        else:

            raise ValueError(
                f"Unknown optimizer: {name}"
            )

        for bar, value in zip(
            bars,
            values
        ):
            bar.set_height(
                float(value)
            )

        max_abs = max(
            1e-6,
            float(
                np.max(
                    np.abs(values)
                )
            ) * 1.30
        )

        ax1.set_ylim(
            -max_abs,
            max_abs
        )

        text = (
            f"{name}\n"
            f"{'─' * 31}\n"
            f"Step       : {int(row['step'])}\n"
            f"Loss       : {row['loss']:.6f}\n"
            f"Weight     : {row['w']:.6f}\n"
            f"Bias       : {row['b']:.6f}\n"
            f"Gradient w : {grad_w:+.6f}\n"
            f"Gradient b : {grad_b:+.6f}\n"
            f"Rule       : {rule}\n"
            f"{extra}"
        )

        info.set_text(text)

        return (
            [path_line, path_point]
            + list(bars)
            + [info]
        )

    anim = FuncAnimation(
        fig,
        update,
        frames=len(h),
        interval=interval,
        blit=False,
        repeat=True
    )

    plt.close(fig)

    return HTML(
        anim.to_jshtml()
    )

In [ ]:
display(mechanics_animation("SGD"))

In [ ]:
display(mechanics_animation("AdaGrad"))

In [ ]:
display(mechanics_animation("RMSProp"))

In [ ]:
display(mechanics_animation("Adam"))

In [ ]:
results = []

for name, model in models.items():

    pred_s = model.predict(
        X_test_s,
        verbose=0
    )

    pred = y_scaler.inverse_transform(
        pred_s
    )

    y_true = y_test

    results.append({
        "Optimizer": name,
        "Final Train MSE (scaled)": histories[name]["loss"].iloc[-1],
        "Test MAE": mean_absolute_error(
            y_true,
            pred
        ),
        "Test RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                pred
            )
        ),
        "Test R²": r2_score(
            y_true,
            pred
        ),
        "Final w": model.get_weights()[0].ravel()[0],
        "Final b": model.get_weights()[1].ravel()[0]
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("Test RMSE")
    .reset_index(drop=True)
)

display(
    results_df.round(5)
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 6)
)

x = np.arange(
    len(results_df)
)

bars = ax.bar(
    x,
    results_df["Test RMSE"],
    color=[
        OPT_COLORS[o]
        for o in results_df["Optimizer"]
    ],
    width=0.65
)

ax.set_xticks(x)
ax.set_xticklabels(
    results_df["Optimizer"]
)

ax.set_ylabel("Test RMSE")
ax.set_title(
    "Final Test RMSE Comparison"
)

for bar, value in zip(
    bars,
    results_df["Test RMSE"]
):
    ax.text(
        bar.get_x()
        + bar.get_width()/2,
        bar.get_height(),
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontweight="bold"
    )

plt.show()

In [ ]:
x_line = np.linspace(
    X.min(),
    X.max(),
    300,
    dtype=np.float32
).reshape(-1, 1)

x_line_s = x_scaler.transform(
    x_line
).astype(np.float32)

fig, ax = plt.subplots(
    figsize=(13, 8)
)

ax.scatter(
    X_train.ravel(),
    y_train.ravel(),
    s=55,
    alpha=0.48,
    label="Train"
)

ax.scatter(
    X_test.ravel(),
    y_test.ravel(),
    s=85,
    alpha=0.88,
    marker="D",
    label="Test"
)

for name, model in models.items():

    pred_line_s = model.predict(
        x_line_s,
        verbose=0
    )

    pred_line = y_scaler.inverse_transform(
        pred_line_s
    ).ravel()

    ax.plot(
        x_line.ravel(),
        pred_line,
        linewidth=3.2,
        color=OPT_COLORS[name],
        label=name
    )

ax.set_title(
    "Learned Linear Function on the Original Data Scale"
)

ax.set_xlabel("Original x")
ax.set_ylabel("Original y")

ax.legend()

plt.show()